# 01.2 Roofline Model for LLM InferenceThis notebook constructs GPU roofline models, calculates arithmetic intensity fortransformer operations, and visualizes memory-bound vs compute-bound regimesacross model sizes and batch sizes.

In [ ]:
import syssys.path.insert(0, '../../..')import numpy as npimport matplotlib.pyplot as pltfrom matplotlib.patches import FancyArrowPatchfrom utils import roofline, gpu_info%matplotlib inlineplt.rcParams['figure.dpi'] = 120plt.rcParams['font.size'] = 11

## 1. GPU Hardware SpecificationsDefine specs for common inference GPUs. The roofline is determined by two numbers:peak compute (TFLOPS) and memory bandwidth (GB/s).

In [ ]:
# GPU specs: (name, FP16 TFLOPS, memory BW GB/s, VRAM GB)GPUS = {    "A10G":  gpu_info.GPUInfo("A10G",  24, 125, 600, 125e3/600),    "A100":  gpu_info.GPUInfo("A100",  80, 312, 2039, 312e3/2039),    "H100":  gpu_info.GPUInfo("H100",  80, 990, 3350, 990e3/3350),}for name, g in GPUS.items():    print(f"{name}: {g.tflops_fp16} TFLOPS, {g.bw_gbs} GB/s, ridge={g.ridge_point:.1f} FLOP/byte")

## 2. Constructing the RooflineThe roofline has two regimes:- **Memory-bound** (left of ridge): Performance = Bandwidth × Arithmetic Intensity- **Compute-bound** (right of ridge): Performance = Peak TFLOPSRidge point = Peak Compute / Memory Bandwidth

In [ ]:
def plot_roofline_basic(gpu, ax=None):    """Plot roofline for a single GPU."""    if ax is None:        fig, ax = plt.subplots(figsize=(10, 6))        ai = np.logspace(-1, 4, 500)    mem_roof = gpu.bw_gbs * ai  # GFLOP/s    compute_roof = gpu.tflops_fp16 * 1000  # GFLOP/s    achieved = np.minimum(mem_roof, compute_roof)        ax.loglog(ai, achieved, 'k-', lw=2.5)    ax.axvline(gpu.ridge_point, color='gray', ls='--', alpha=0.6,               label=f'Ridge: {gpu.ridge_point:.0f} FLOP/byte')    ax.fill_between(ai, 0.1, achieved, where=(ai < gpu.ridge_point),                    alpha=0.08, color='red')    ax.fill_between(ai, 0.1, achieved, where=(ai >= gpu.ridge_point),                    alpha=0.08, color='blue')    ax.set_xlabel('Arithmetic Intensity (FLOP/byte)')    ax.set_ylabel('Performance (GFLOP/s)')    ax.set_xlim(0.1, 1e4)    ax.set_ylim(1, compute_roof * 3)    ax.legend(fontsize=10)    ax.grid(True, alpha=0.3)    return axfig, ax = plt.subplots(figsize=(10, 6))plot_roofline_basic(GPUS['A100'], ax)ax.set_title('A100 80GB Roofline Model (FP16)')plt.tight_layout()plt.show()

## 3. Multi-GPU Roofline ComparisonCompare rooflines across GPU generations to see how the ridge point shifts.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))colors = {'A10G': 'tab:green', 'A100': 'tab:blue', 'H100': 'tab:red'}for name, g in GPUS.items():    ai = np.logspace(-1, 4, 500)    mem_roof = g.bw_gbs * ai    compute_roof = g.tflops_fp16 * 1000    achieved = np.minimum(mem_roof, compute_roof)    ax.loglog(ai, achieved, lw=2.5, color=colors[name],              label=f'{name} (ridge={g.ridge_point:.0f})')    ax.axvline(g.ridge_point, color=colors[name], ls=':', alpha=0.4)ax.set_xlabel('Arithmetic Intensity (FLOP/byte)')ax.set_ylabel('Performance (GFLOP/s)')ax.set_title('Roofline Comparison: A10G vs A100 vs H100')ax.set_xlim(0.1, 1e4)ax.legend(fontsize=11)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()

## 4. Arithmetic Intensity of LLM OperationsFor decode (batch=B, model with P parameters in FP16):- FLOPs per token ≈ 2P × B (one matmul per weight, amortized across batch)- Bytes read ≈ 2P (read all weights once, shared across batch)**AI_decode = B** (approximately)For prefill (sequence length S):- FLOPs ≈ 2P × S- Bytes ≈ 2P (weights dominate for large S)**AI_prefill ≈ S**

In [ ]:
def arithmetic_intensity_decode(batch_size, params_b, dtype_bytes=2):    """Arithmetic intensity for LLM decode."""    flops = 2 * params_b * 1e9 * batch_size    bytes_read = params_b * 1e9 * dtype_bytes  # read all weights once    return flops / bytes_readdef arithmetic_intensity_prefill(seq_len, params_b, dtype_bytes=2):    """Arithmetic intensity for LLM prefill."""    flops = 2 * params_b * 1e9 * seq_len    bytes_read = params_b * 1e9 * dtype_bytes    return flops / bytes_read# Example: Llama 8B decodemodels = {'Llama-8B': 8, 'Llama-70B': 70, 'Llama-405B': 405}batch_sizes = [1, 4, 16, 64, 128, 256]print('Arithmetic Intensity (FLOP/byte) - Decode')print(f'{"Batch":>6}', end='')for m in models:    print(f'{m:>12}', end='')print()print('-' * 42)for b in batch_sizes:    print(f'{b:>6}', end='')    for name, p in models.items():        ai = arithmetic_intensity_decode(b, p)        print(f'{ai:>12.1f}', end='')    print()print(f'A100 ridge point: {GPUS["A100"].ridge_point:.0f} FLOP/byte')print('→ Need batch ≈ 153 to become compute-bound on A100')

## 5. Decode Batch Size on the RooflineVisualize how increasing batch size moves decode operations up the roofline.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))gpu = GPUS['A100']plot_roofline_basic(gpu, ax)ax.set_title('A100 Roofline: LLM Decode at Various Batch Sizes (Llama 8B)')batch_sizes = [1, 4, 16, 32, 64, 128, 256]for b in batch_sizes:    ai = arithmetic_intensity_decode(b, 8)    perf = min(gpu.bw_gbs * ai, gpu.tflops_fp16 * 1000)  # GFLOP/s    ax.scatter(ai, perf, s=120, zorder=5, edgecolors='white', lw=0.8)    ax.annotate(f'B={b}', (ai, perf), textcoords='offset points',                xytext=(6, 6), fontsize=9)ax.set_xlim(0.5, 2000)plt.tight_layout()plt.show()

## 6. Model Size Impact on Roofline PositionLarger models have the same arithmetic intensity (AI ≈ batch_size) but differentabsolute performance because they do more total FLOPs per token.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)gpu = GPUS['A100']for idx, (model_name, params) in enumerate(models.items()):    ax = axes[idx]    plot_roofline_basic(gpu, ax)    ax.set_title(f'{model_name} on A100')        for b in [1, 8, 32, 128]:        ai = arithmetic_intensity_decode(b, params)        perf = min(gpu.bw_gbs * ai, gpu.tflops_fp16 * 1000)        ax.scatter(ai, perf, s=100, zorder=5, edgecolors='white', lw=0.8)        ax.annotate(f'B={b}', (ai, perf), textcoords='offset points',                    xytext=(5, 5), fontsize=8)    ax.set_xlim(0.5, 2000)plt.suptitle('Decode Arithmetic Intensity Across Model Sizes', fontsize=13, y=1.02)plt.tight_layout()plt.show()

## 7. Prefill vs Decode: Two Different WorldsPrefill processes S tokens in parallel → AI ≈ S (compute-bound for long prompts).Decode generates 1 token at a time → AI ≈ B (memory-bound unless huge batch).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))gpu = GPUS['A100']plot_roofline_basic(gpu, ax)ax.set_title('A100: Prefill vs Decode Regimes (Llama 8B)')# Decode pointsfor b in [1, 8, 32, 128]:    ai = arithmetic_intensity_decode(b, 8)    perf = min(gpu.bw_gbs * ai, gpu.tflops_fp16 * 1000)    ax.scatter(ai, perf, s=100, marker='o', color='tab:red', zorder=5,               edgecolors='white', lw=0.8)    ax.annotate(f'Decode B={b}', (ai, perf), textcoords='offset points',                xytext=(6, -12), fontsize=8, color='tab:red')# Prefill pointsfor s in [128, 512, 2048, 8192]:    ai = arithmetic_intensity_prefill(s, 8)    perf = min(gpu.bw_gbs * ai, gpu.tflops_fp16 * 1000)    ax.scatter(ai, perf, s=100, marker='^', color='tab:blue', zorder=5,               edgecolors='white', lw=0.8)    ax.annotate(f'Prefill S={s}', (ai, perf), textcoords='offset points',                xytext=(6, 6), fontsize=8, color='tab:blue')ax.set_xlim(0.5, 1e4)plt.tight_layout()plt.show()

## 8. The KV Cache ConstraintBatching increases AI but KV cache grows linearly with batch size.At some point VRAM is exhausted before reaching the ridge point.**KV cache per sequence** = 2 × layers × kv_heads × head_dim × seq_len × dtype_bytes

In [ ]:
def kv_cache_gb(layers, kv_heads, head_dim, seq_len, batch, dtype_bytes=2):    """KV cache size in GB."""    return 2 * layers * kv_heads * head_dim * seq_len * batch * dtype_bytes / 1e9def max_batch_for_vram(vram_gb, model_gb, layers, kv_heads, head_dim, seq_len, overhead_gb=3):    """Max batch size that fits in VRAM."""    available = vram_gb - model_gb - overhead_gb    kv_per_seq = kv_cache_gb(layers, kv_heads, head_dim, seq_len, 1)    return int(available / kv_per_seq)# Llama 8B configcfg = {'layers': 32, 'kv_heads': 8, 'head_dim': 128}seq_lengths = [1024, 2048, 4096, 8192, 16384]print('Max batch size on A100 80GB (Llama 8B FP16, 16GB weights):')print(f'{"SeqLen":>8} {"KV/seq (MB)":>12} {"MaxBatch":>10} {"AI at max":>10} {"Bound":>12}')print('-' * 55)for s in seq_lengths:    kv_seq = kv_cache_gb(**cfg, seq_len=s, batch=1) * 1000  # MB    mb = max_batch_for_vram(80, 16, **cfg, seq_len=s)    ai = arithmetic_intensity_decode(mb, 8)    bound = 'COMPUTE' if ai >= GPUS['A100'].ridge_point else 'MEMORY'    print(f'{s:>8} {kv_seq:>12.0f} {mb:>10} {ai:>10.0f} {bound:>12}')print(f'Ridge point: {GPUS["A100"].ridge_point:.0f} FLOP/byte')print('→ Always memory-bound: KV cache fills VRAM before reaching ridge')

## 9. Visualizing the VRAM-Bounded RooflinePlot achievable region considering KV cache memory limits.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))gpu = GPUS['A100']plot_roofline_basic(gpu, ax)ax.set_title('A100 Roofline: VRAM-Bounded Operating Points (Llama 8B)')colors_seq = {1024: 'tab:blue', 2048: 'tab:orange', 4096: 'tab:green',              8192: 'tab:red', 16384: 'tab:purple'}for s, color in colors_seq.items():    mb = max_batch_for_vram(80, 16, **cfg, seq_len=s)    batches = [b for b in [1, 4, 16, 32, 64, 128, 256, 512] if b <= mb]    ais = [arithmetic_intensity_decode(b, 8) for b in batches]    perfs = [min(gpu.bw_gbs * ai, gpu.tflops_fp16 * 1000) for ai in ais]    ax.plot(ais, perfs, 'o-', color=color, label=f'seq={s} (max_B={mb})',             markersize=6, lw=1.5)    # Mark the max batch point    ax.scatter(ais[-1], perfs[-1], s=200, color=color, marker='*', zorder=6,               edgecolors='black', lw=0.5)ax.set_xlim(0.5, 2000)ax.legend(title='Context Length', fontsize=9)plt.tight_layout()plt.show()

## 10. Quantization Shifts the RooflineQuantization reduces bytes per parameter → same FLOPs with fewer bytes read → higher AI.INT4 gives 4× improvement in arithmetic intensity vs FP16.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))gpu = GPUS['A100']plot_roofline_basic(gpu, ax)ax.set_title('Quantization Impact on Roofline Position (Llama 70B, Batch=8)')dtypes = {'FP16': 2, 'INT8': 1, 'INT4': 0.5}colors_dt = {'FP16': 'tab:red', 'INT8': 'tab:orange', 'INT4': 'tab:green'}batch = 8for dtype_name, dtype_bytes in dtypes.items():    ai = arithmetic_intensity_decode(batch, 70, dtype_bytes)    perf = min(gpu.bw_gbs * ai, gpu.tflops_fp16 * 1000)    ax.scatter(ai, perf, s=150, color=colors_dt[dtype_name], zorder=5,               edgecolors='black', lw=1, label=f'{dtype_name} (AI={ai:.0f})')    ax.annotate(f'{dtype_name}', (ai, perf), textcoords='offset points',                xytext=(8, 8), fontsize=10, fontweight='bold',                color=colors_dt[dtype_name])ax.set_xlim(0.5, 2000)ax.legend(fontsize=11, title='Weight Dtype')plt.tight_layout()plt.show()print('Quantization multiplies effective AI:')for dtype_name, dtype_bytes in dtypes.items():    ai = arithmetic_intensity_decode(batch, 70, dtype_bytes)    print(f'  {dtype_name}: AI = {ai:.1f} FLOP/byte')

## 11. Theoretical Throughput CeilingMax decode tokens/s = Memory Bandwidth / Model Size (bytes)This is the hard ceiling — no software optimization can exceed it.

In [ ]:
def max_tokens_per_sec(bw_gbs, model_gb):    """Theoretical max decode throughput (batch=1)."""    return bw_gbs / model_gbprint('Theoretical Max Decode Throughput (tokens/s, batch=1)')print(f'{"":>12}', end='')for g_name in GPUS:    print(f'{g_name:>10}', end='')print()print('-' * 42)model_configs = [    ('8B FP16', 16), ('8B INT8', 8), ('8B INT4', 4),    ('70B FP16', 140), ('70B INT4', 35),]for m_name, m_gb in model_configs:    print(f'{m_name:>12}', end='')    for g_name, g in GPUS.items():        if g.vram_gb >= m_gb * 1.1:            tps = max_tokens_per_sec(g.bw_gbs, m_gb)            print(f'{tps:>10.0f}', end='')        else:            print(f'{"OOM":>10}', end='')    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))# Left: throughput by GPU for 8B modelax = axes[0]dtypes_8b = {'FP16 (16GB)': 16, 'INT8 (8GB)': 8, 'INT4 (4GB)': 4}x = np.arange(len(GPUS))width = 0.25for i, (dt_name, m_gb) in enumerate(dtypes_8b.items()):    tps_vals = [g.bw_gbs / m_gb for g in GPUS.values()]    ax.bar(x + i*width, tps_vals, width, label=dt_name)ax.set_xticks(x + width)ax.set_xticklabels(GPUS.keys())ax.set_ylabel('Max Tokens/s (batch=1)')ax.set_title('Llama 8B: Theoretical Decode Throughput')ax.legend()ax.grid(axis='y', alpha=0.3)# Right: throughput by batch size on A100ax = axes[1]batches = np.arange(1, 129)gpu = GPUS['A100']# Effective throughput scales with batch until compute-boundfor dt_name, dt_bytes in [('FP16', 2), ('INT8', 1), ('INT4', 0.5)]:    model_gb = 8 * dt_bytes    base_tps = gpu.bw_gbs / model_gb  # per-sequence at batch=1    # Total throughput = min(batch * base_tps, compute_ceiling)    compute_ceil = gpu.tflops_fp16 * 1e3 / (2 * 8)  # GFLOP/s / GFLOP-per-token    total_tps = np.minimum(batches * base_tps, compute_ceil)    ax.plot(batches, total_tps, lw=2, label=f'{dt_name}')ax.set_xlabel('Batch Size')ax.set_ylabel('Total Tokens/s')ax.set_title('A100: Aggregate Throughput vs Batch (Llama 8B)')ax.legend()ax.grid(alpha=0.3)plt.tight_layout()plt.show()

## 12. Bandwidth Utilization AnalysisReal systems achieve 60-80% of theoretical bandwidth due to:- Memory access patterns (non-coalesced reads)- Kernel launch overhead- Tensor parallelism communication- PagedAttention indirection

In [ ]:
def effective_throughput(gpu, model_gb, batch, efficiency=0.75):    """Estimate real-world throughput with efficiency factor."""    theoretical = gpu.bw_gbs / model_gb * batch    return theoretical * efficiencyfig, ax = plt.subplots(figsize=(10, 6))gpu = GPUS['A100']batches = np.array([1, 2, 4, 8, 16, 32, 64, 128])efficiencies = [0.6, 0.7, 0.8, 0.9, 1.0]for eff in efficiencies:    tps = [effective_throughput(gpu, 16, b, eff) for b in batches]    style = '-' if eff < 1.0 else '--'    ax.plot(batches, tps, style, lw=2, label=f'{eff:.0%} BW utilization')ax.set_xlabel('Batch Size')ax.set_ylabel('Tokens/s (total)')ax.set_title('A100 Decode Throughput: Llama 8B FP16 at Various BW Efficiencies')ax.legend()ax.set_xscale('log', base=2)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print('Typical real-world BW utilization:')print('  vLLM:        70-80%')print('  TensorRT-LLM: 75-85%')print('  Custom CUDA:  80-90%')

## Key Takeaways1. **LLM decode is always memory-bound** — AI ≈ batch_size, ridge point ≈ 150+ FLOP/byte2. **KV cache prevents reaching the ridge** — VRAM fills before batch is large enough3. **Quantization is a bandwidth optimization** — INT4 gives 4× effective AI improvement4. **Prefill is compute-bound** for sequences > ridge_point tokens5. **Select GPUs by bandwidth**, not just VRAM or TFLOPS6. **Real systems achieve 60-80%** of theoretical bandwidth ceiling7. **GPU utilization is misleading** — 30% compute util with 75% BW util is optimal for decode